# 🔬 DermaFusion-AI — Kaggle Training Notebook
## EVA-02 Large + ConvNeXt V2 | Dual-Branch Fusion | SOTA 2026

### ⚙️ Before Running:
1. **Add Data** (right panel) — attach 4 datasets: ISIC 2024, SIIM-ISIC Melanoma, Skin Lesion Images, HAM10000 ✅
2. Set **Accelerator → GPU T4 x2** in the right panel
3. Set **Persistence → Files** so weights survive between sessions
4. Run cells **top to bottom**

## Step 0: Inspect All Available Dataset Paths

In [ ]:
import os

def explore(path, depth=0, max_depth=2):
    if depth > max_depth or not os.path.exists(path):
        return
    items = sorted(os.listdir(path))
    for item in items:
        full = os.path.join(path, item)
        kind = '📁' if os.path.isdir(full) else '📄'
        print('  ' * depth + f'{kind} {item}')
        if os.path.isdir(full):
            explore(full, depth + 1, max_depth)

print('📂 /kaggle/input/ structure:')
explore('/kaggle/input', max_depth=2)

## Step 1: Clone Your GitHub Repo

In [ ]:
import os

if not os.path.exists('/kaggle/working/DermaFusion-AI'):
    !git clone https://github.com/ai-with-abdullah/DermaFusion-AI.git /kaggle/working/DermaFusion-AI
else:
    print('Repo already cloned — pulling latest')
    !cd /kaggle/working/DermaFusion-AI && git pull

os.chdir('/kaggle/working/DermaFusion-AI')
print('Working directory:', os.getcwd())

## Step 2: Install Dependencies

In [ ]:
!pip install -q timm>=0.9.12 albumentations>=1.3.1 opencv-python-headless scikit-learn scipy tqdm
print('✅ Dependencies installed')

## Step 3: Verify GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'✅ GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')
else:
    print('❌ No GPU — go to Settings > Accelerator > GPU T4 x2')

## Step 4: Link Datasets → data/ folder
Kaggle puts your data under `/kaggle/input/competitions/` and `/kaggle/input/datasets/`.
This cell searches both locations and auto-links them to the paths the training code expects.

In [ ]:
import os

data_dir = '/kaggle/working/DermaFusion-AI/data'
os.makedirs(data_dir, exist_ok=True)

# Search both competitions and datasets subdirs
search_roots = [
    '/kaggle/input/competitions',
    '/kaggle/input/datasets',
    '/kaggle/input',  # fallback: flat layout
]

def find_folder(keywords):
    """Search all roots for a folder whose name contains any keyword."""
    for root in search_roots:
        if not os.path.exists(root):
            continue
        for folder in os.listdir(root):
            full = os.path.join(root, folder)
            if os.path.isdir(full) and any(kw in folder.lower() for kw in keywords):
                return full
    return None

def link(src, dst_name, label):
    dst = os.path.join(data_dir, dst_name)
    if os.path.exists(dst):
        print(f'✅ {label}: already linked')
        return True
    if src and os.path.exists(src):
        os.symlink(src, dst)
        print(f'✅ {label}: linked from {src}')
        return True
    print(f'❌ {label}: NOT FOUND')
    return False

# ── HAM10000 ─────────────────────────────────────────────────────────────────
# Expects: data/ham10000/HAM10000_images_part_1/, data/ham10000/HAM10000_metadata.csv
ham_src = find_folder(['ham10000', 'skin-cancer-mnist'])
link(ham_src, 'ham10000', 'HAM10000')

# ── ISIC 2019 ────────────────────────────────────────────────────────────────
# Expects: data/isic_2019/ISIC_2019_Training_Input/, data/isic_2019/ISIC_2019_Training_GroundTruth.csv
isic19_src = find_folder(['skin-lesion-images', 'isic-2019', 'isic2019'])
link(isic19_src, 'isic_2019', 'ISIC 2019')

# ── ISIC 2020 ────────────────────────────────────────────────────────────────
# Expects: data/isic_2020/train/ (images), data/isic_2020/train.csv
isic20_src = find_folder(['siim-isic', 'melanoma-classification', 'isic-2020'])
link(isic20_src, 'isic_2020', 'ISIC 2020')

# ── ISIC 2024 ────────────────────────────────────────────────────────────────
# Expects: data/isic_2024/train-image/image/, data/isic_2024/train-metadata.csv
isic24_src = find_folder(['isic-2024', 'isic2024', 'skin-cancer-detection-3d'])
link(isic24_src, 'isic_2024', 'ISIC 2024')

print('\n📁 data/ contents:', os.listdir(data_dir))

## Step 5: Phase 1 — Train Segmentation Model (25 epochs)
Trains **Swin-UNet** to produce lesion masks. Must run before classifier.

In [ ]:
import os
os.chdir('/kaggle/working/DermaFusion-AI')
!python train_segmentation.py 2>&1 | tee /kaggle/working/train_segmentation.log
print('\n✅ Segmentation training complete!')

## Step 6: Phase 2 — Train Classifier (25 epochs)
Trains **EVA-02 Large + ConvNeXt V2** dual-branch fusion classifier.

In [ ]:
import os
os.chdir('/kaggle/working/DermaFusion-AI')
!python train_classifier.py 2>&1 | tee /kaggle/working/train_classifier.log
print('\n✅ Classifier training complete!')

## Step 7: Save Weights to Kaggle Output

In [ ]:
import shutil, os
weights_src = '/kaggle/working/DermaFusion-AI/outputs/weights/'
if os.path.exists(weights_src):
    for f in os.listdir(weights_src):
        if f.endswith('.pth'):
            shutil.copy(os.path.join(weights_src, f), '/kaggle/working/')
            size_mb = os.path.getsize(f'/kaggle/working/{f}') / 1e6
            print(f'✅ Saved: {f}  ({size_mb:.0f} MB)')
    print('\n📦 Download weights from the Output tab on the right.')
else:
    print('❌ No weights found — did training complete?')

## Step 8: Full Evaluation (Metrics + GradCAM++ XAI)

In [ ]:
import os
os.chdir('/kaggle/working/DermaFusion-AI')
!python evaluate.py 2>&1 | tee /kaggle/working/evaluate.log
print('\n✅ Evaluation complete!')